# Classification de texte : régression logistique multinomiale from scratch

**Série NLP — carnet 11.** Ce carnet construit, entraîne et évalue un classifieur de texte **discriminatif** en C# avec MathNet : softmax, entropie croisée et descente de gradient avec régularisation L2. Aucun framework de descente de gradient n'est utilisé — `scikit-learn` intervient uniquement comme **juge extérieur**, à la fin.

## Où ce carnet se situe dans la série

| Carnet | Objet | Modèle |
|---|---|---|
| 06 | recherche d'information | TF-IDF, BM25 — pondération, pas de classification |
| 07 à 09 | plongements de mots | co-occurrence/PPMI/SVD, word2vec, fastText |
| 10 | classification de texte | Naive Bayes — **génératif** |
| **11** | classification de texte | régression logistique — **discriminatif** |

La différence entre 10 et 11 n'est pas un détail d'implémentation.

- Naive Bayes **modélise** `P(mots | classe)` et suppose les mots **indépendants** conditionnellement à la classe. L'hypothèse est fausse sur du texte réel, et elle est fausse d'une manière qui **compte** : des mots corrélés sont comptés plusieurs fois.
- La régression logistique modélise **directement** `P(classe | mots)` et ne suppose aucune indépendance. Elle apprend des **poids** qui se compensent entre eux.

Le prix de cette différence est une optimisation itérative au lieu d'un simple comptage. C'est ce prix que ce carnet rend visible : une courbe de convergence, un balayage de régularisation, et une comparaison de bout en bout avec l'implémentation de référence.

## Ce que vous saurez faire à la fin

1. écrire la fonction softmax et montrer pourquoi on retire le maximum avant l'exponentielle ;
2. dériver le gradient de l'entropie croisée softmax et vérifier qu'il tient en `P - Y` ;
3. entraîner un modèle par descente de gradient pleine passe et lire sa convergence ;
4. mesurer une précision **hors échantillon** et la comparer à une baseline ;
5. expliquer ce qu'un balayage de `lambda` révèle, et où se situe le compromis.

## Plan

1. Pourquoi le corpus jouet ne suffit pas
2. Représentation : sac de mots, fréquences relatives
3. Le modèle : softmax, entropie croisée, gradient L2 — et la précision hors échantillon mesurée
4. Cross-check SOTA contre `scikit-learn`
5. Ce que la régularisation fait — et ce qu'elle ne fait pas
6. Synthèse
7. Exercices, puis références

Les titres numérotés du carnet suivent exactement ce plan ; la précision hors échantillon est la dernière lecture de la section 3, elle n'a pas de rubrique à elle.
**Navigation** : [Précédent : Naive Bayes](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) | [Index NLP](README.md) | [Version Python](11-Classement-Texte-RegressionLogistique-FromScratch.ipynb)

**Prérequis** : C#, algèbre linéaire et probabilités. **Durée** : 90 minutes.

**Prérequis du témoin** : installer CPython, NumPy et scikit-learn, puis définir `PYTHONNET_PYDLL` vers la bibliothèque partagée Python avant de lancer Jupyter. Définir `PYTHONHOME` et `PYTHONPATH` si nécessaire pour sélectionner l’interpréteur et les paquets. Python.NET ne sert qu’à la comparaison scikit-learn ; les calculs principaux sont en C#.


In [1]:
#r "nuget: MathNet.Numerics, 5.0.0"
#r "nuget: pythonnet, 3.0.5"
using System;
using System.Linq;
using System.Collections.Generic;
using System.Text.RegularExpressions;
using MathNet.Numerics.LinearAlgebra;
using MathNet.Numerics.Distributions;
using Python.Runtime;
var corpus = new Dictionary<string,string[]> {
    ["sport"] = new [] {
        "l'equipe du stade remporte le match sur un dernier but",
        "l'arbitre siffle la fin du match dans le stade",
        "le coach motive l'equipe avant le match decisif",
        "le gardien arrete le penalty au dernier moment",
        "les joueurs celebrent la victoire dans le stade",
        "le buteur marque le but decisif du match",
        "le score final du match est historique au stade",
        "l'equipe entree au stade pour le match de la saison"
    },
    ["tech"] = new [] {
        "le developp corrige un bug dans le logiciel",
        "le serveur renvoie une erreur dans le log du systeme",
        "la puce integre un accelerateur dans le processeur",
        "le framework compile le code en quelques secondes",
        "l'application consomme la memoire au demarrage du systeme",
        "l'algorithme accelere le calcul dans le logiciel",
        "l'interface du logiciel interroge le serveur du systeme",
        "le code du systeme compile sans erreur"
    },
    ["science"] = new [] {
        "l'experience mesure la constante avec une grande precision",
        "le paper publie les resultats de l'observation",
        "l'hypothese est confirmee par les donnees recueillies",
        "le chercheur analyse les echantillons au microscope",
        "la theorie predit le comportement du systeme physique",
        "l'article rapporte une mesure de l'energie",
        "les mesures de l'experience valident le modele theorique",
        "l'observation confirme le resultat de l'experience"
    },
    ["finance"] = new [] {
        "la bourse enregistre une hausse des indices ce matin",
        "la banque annonce un resultat net en hausse",
        "l'action cote plus haut a la fin de la seance",
        "l'investisseur choisit l'obligation pour le credit",
        "le taux d'interet guide la decision d'emprunt",
        "le portefeuille subit une baisse des valeurs",
        "le dividende est verse aux actionnaires du marche",
        "le marche anticipe une revision des taux"
    }
};
var categories = corpus.Keys.ToArray();
string[] Tokeniser(string texte) => Regex.Matches(texte.ToLowerInvariant(), "[a-z]+").Select(x => x.Value).ToArray();
Console.WriteLine($".NET {Environment.Version}, MathNet {typeof(Normal).Assembly.GetName().Version}, Python.NET {typeof(PythonEngine).Assembly.GetName().Version}");
Console.WriteLine($"Corpus jouet : {corpus.Sum(x=>x.Value.Length)} documents, {categories.Length} classes ; exemple : {corpus["sport"][0]}");

Installed Packages MathNet.Numerics, 5.0.0 pythonnet, 3.0.5

.NET 10.0.14, MathNet 5.0.0.0, Python.NET 3.0.5.0


Corpus jouet : 32 documents, 4 classes ; exemple : l'equipe du stade remporte le match sur un dernier but


## 1. Pourquoi le corpus jouet ne suffit pas

Le carnet 10 travaille sur 32 documents courts, 8 par catégorie. Ce petit corpus offre un banc de test très limité : huit documents seulement étaient réservés au test dans le carnet 10. Une telle mesure ne suffit pas à caractériser la généralisation.

Ce carnet garde ce corpus comme **socle de vocabulaire** — c'est lui qui donne au duo 10/11 la même langue — mais construit à partir de lui un banc où les classes se **chevauchent** :

- le vocabulaire reste **commun** aux quatre catégories, y compris les mots grammaticaux qui n'appartiennent à aucune ;
- chaque catégorie en sur-représente une partie, sans jamais l'isoler ;
- un document est un **mélange** : ses mots sont tirés d'une distribution `lambda * P(categorie) + (1 - lambda) * P(autre categorie)` ;
- `lambda` suit une loi `Beta(5, 2)`, dont la moyenne vaut `alpha / (alpha + beta) = 5/7`, soit `0,714` : un document **penche** vers sa classe, sans jamais l'isoler. Les documents où `lambda < 0.65` sont comptés et affichés — ce sont eux qui rendent le banc difficile, et le carnet les mesure au lieu de les cacher.

Le générateur est **déterministe** (graine fixée ; le générateur C# diffère de NumPy, donc les documents et résultats peuvent différer) : le banc est reproductible, et les mesures de ce carnet sont comparables d'une exécution à l'autre. Le découpage apprentissage/test est fait **avant** tout entraînement, une seule fois.

In [2]:
var vocabulaire = corpus.Values.SelectMany(x => x).SelectMany(Tokeniser).Distinct().OrderBy(x=>x,StringComparer.Ordinal).ToArray();
var position = vocabulaire.Select((w,i)=>(w,i)).ToDictionary(x=>x.w,x=>x.i);
double[] Distribution(string[] docs) {
    var counts = new double[vocabulaire.Length];
    foreach(var word in docs.SelectMany(Tokeniser)) counts[position[word]]++;
    double total=counts.Sum(); return counts.Select(x=>(x+0.05)/(total+0.05*counts.Length)).ToArray();
}
var distributions = categories.Select(c=>Distribution(corpus[c])).ToArray();
var rng = new Random(20261029); var beta = new Beta(5,2,rng);
var documents = new List<string>(); var etiquettes = new List<int>(); var melanges = new List<double>();
for(int c=0;c<categories.Length;c++) for(int j=0;j<60;j++) {
    var autres=Enumerable.Range(0,4).Where(x=>x!=c).ToArray(); int autre=autres[rng.Next(3)];
    double lam=beta.Sample(); melanges.Add(lam); int length=rng.Next(25,41);
    var cumul=new double[vocabulaire.Length]; double somme=0;
    for(int i=0;i<cumul.Length;i++) { somme+=lam*distributions[c][i]+(1-lam)*distributions[autre][i]; cumul[i]=somme; }
    var mots=new List<string>();
    for(int i=0;i<length;i++) { double u=rng.NextDouble()*somme; int ix=Array.FindIndex(cumul,x=>x>=u); mots.Add(vocabulaire[ix]); }
    documents.Add(string.Join(" ",mots)); etiquettes.Add(c);
}
var rngSplit=new Random(20261030); var ordre=Enumerable.Range(0,documents.Count).ToArray();
for(int i=ordre.Length-1;i>0;i--) { int j=rngSplit.Next(i+1); (ordre[i],ordre[j])=(ordre[j],ordre[i]); }
var idxTrain=ordre.Take(144).ToArray(); var idxTest=ordre.Skip(144).ToArray();
Console.WriteLine($"Banc : {documents.Count} documents, vocabulaire {vocabulaire.Length} ; train {idxTrain.Length}, test {idxTest.Length}");
Console.WriteLine($"Ambigus lambda < 0.65 : {melanges.Count(x=>x<0.65)} ; exemple : {documents[0]}");

Banc : 240 documents, vocabulaire 138 ; train 144, test 96


Ambigus lambda < 0.65 : 86 ; exemple : le systeme serveur l l le dernier application corrige equipe le le le penalty gardien application du compile penalty la au but calcul marque le le score le equipe logiciel decisif le du le un


## 2. Représentation : sac de mots, fréquences relatives

Comme au carnet 06, chaque document devient un vecteur sur un vocabulaire fixe. Deux choix méritent d'être explicités :

- **fréquences relatives** plutôt que comptes bruts : un document long ne doit pas peser plus lourd dans la fonction de perte du seul fait de sa longueur ;
- **pas de TF-IDF ici** : le TF-IDF est le sujet du carnet 06. Le but de ce carnet est de faire porter la discrimination au **modèle**, pas à la pondération des entrées.

La baseline est la **classe majoritaire** : prédire toujours la catégorie la plus fréquente du jeu d'apprentissage. C'est le plancher qu'un classifieur doit dépasser pour avoir appris quelque chose. Sur un banc équilibré elle vaut 25 % — un modèle à 25 % n'a rien appris du texte, il a appris le compte des classes.

In [3]:
var M = Matrix<double>.Build; var V = Vector<double>.Build;
Matrix<double> ConstruireMatriceTokens(IEnumerable<IEnumerable<string>> docs, string[] vocab) {
    var rows=docs.Select(tokens=>tokens.ToArray()).ToArray();
    var pos=vocab.Select((w,i)=>(w,i)).ToDictionary(x=>x.w,x=>x.i);
    var result=M.Dense(rows.Length,vocab.Length);
    for(int i=0;i<rows.Length;i++) {
        foreach(var word in rows[i]) if(pos.TryGetValue(word,out int j)) result[i,j]++;
        double total=result.Row(i).Sum(); if(total>0) result.SetRow(i,result.Row(i)/total);
    }
    return result;
}
Matrix<double> ConstruireMatrice(IEnumerable<string> docs, string[] vocab) =>
    ConstruireMatriceTokens(docs.Select(Tokeniser),vocab);

var X=ConstruireMatrice(documents,vocabulaire);
Matrix<double> Selection(Matrix<double> x, int[] indices) => M.DenseOfRowVectors(indices.Select(i=>x.Row(i)));
var XTrain=Selection(X,idxTrain); var XTest=Selection(X,idxTest);
var yTrain=idxTrain.Select(i=>etiquettes[i]).ToArray(); var yTest=idxTest.Select(i=>etiquettes[i]).ToArray();
var YTrain=M.Dense(yTrain.Length,4,(i,j)=>yTrain[i]==j?1:0);
var majority=yTrain.GroupBy(x=>x).OrderByDescending(g=>g.Count()).ThenBy(g=>categories[g.Key],StringComparer.Ordinal).First().Key;
double baseline=(double)yTest.Count(y=>y==majority)/yTest.Length;
var fixture=ConstruireMatrice(new[]{"b b inconnu","a"},new[]{"b"});
if(fixture[0,0]!=1 || fixture[1,0]!=0 || ConstruireMatrice(new[]{"xyz"},new[]{"xyz"})[0,0]!=1) throw new Exception("Fixture vocabulaire incorrecte");
var pretokenise=ConstruireMatriceTokens(new[]{new[]{"a","b","a_b"}},new[]{"a","b","a_b"});
if (pretokenise.Row(0).Any(v => Math.Abs(v-1.0/3)>1e-12)) throw new Exception("Fixture bigramme incorrecte");
Console.WriteLine($"Matrice {X.RowCount} x {X.ColumnCount} ; densité {(double)X.Enumerate().Count(v=>v!=0)/(X.RowCount*X.ColumnCount):P1}");
Console.WriteLine($"Ligne 0 somme {X.Row(0).Sum():F6} ; baseline {baseline:P2}");
Console.WriteLine("Vocabulaire alternatif, OOV et bigramme pré-tokenisé : contrôles validés");

Matrice 240 x 138 ; densité 16.6 %


Ligne 0 somme 1.000000 ; baseline 20.83 %


Vocabulaire alternatif, OOV et bigramme pré-tokenisé : contrôles validés


## 3. Le modèle : softmax, entropie croisée, gradient L2

Pour un document de caractéristiques `x` (V dimensions) et `K` catégories, le modèle calcule un score par catégorie, puis le normalise en probabilités :

$$z = xW + b, \qquad P(k \mid x) = \frac{e^{z_k}}{\sum_j e^{z_j}}$$

La perte est l'entropie croisée moyenne, plus un terme L2 sur les poids :

$$L(W, b) = -\frac{1}{n}\sum_i \log P(y_i \mid x_i) + \frac{\lambda}{2}\lVert W \rVert^2$$

Le gradient a une forme remarquablement simple. En posant `dZ = (P - Y) / n` :

$$\frac{\partial L}{\partial W} = X^\top \, dZ + \lambda W, \qquad \frac{\partial L}{\partial b} = \sum_i dZ_i$$

Le biais `b` **n'est pas** régularisé : il ne multiplie aucune entrée, le pénaliser reviendrait à décourager les classes rares sans aucune raison.

Deux précautions d'implémentation, et elles ne sont pas cosmétiques :

- **stabilité numérique** : on retire le maximum de `z` avant l'exponentielle. La softmax est invariante à un décalage constant, donc le résultat est identique, mais l'exponentielle ne déborde plus ;
- **`log` écrêté** : `log(P)` utilise une probabilité bornée inférieurement à `1e-12`. Sans cela, une probabilité nulle sur la bonne classe envoie la perte à l'infini et la descente à `NaN`.

Notez ce que ce gradient dit : l'erreur `P - Y` est **la seule** information qui remonte. Là où Naive Bayes comptait des occurrences, ici chaque document pousse les poids dans la direction qui corrige sa propre probabilité.

In [4]:
Matrix<double> Softmax(Matrix<double> z) {
    var p=M.Dense(z.RowCount,z.ColumnCount);
    for(int i=0;i<z.RowCount;i++) { double max=z.Row(i).Maximum(), sum=0;
        for(int j=0;j<z.ColumnCount;j++) { p[i,j]=Math.Exp(z[i,j]-max); sum+=p[i,j]; }
        for(int j=0;j<z.ColumnCount;j++) p[i,j]/=sum;
    } return p;
}
Matrix<double> Scores(Matrix<double> x, Matrix<double> w, Vector<double> b) {
    var z=x*w; for(int i=0;i<z.RowCount;i++) z.SetRow(i,z.Row(i)+b); return z;
}
double Perte(Matrix<double> x, Matrix<double> y, Matrix<double> w, Vector<double> b, double lam) {
    var p=Softmax(Scores(x,w,b)); double loss=0;
    for(int i=0;i<p.RowCount;i++) for(int j=0;j<p.ColumnCount;j++) if(y[i,j]>0) loss-=Math.Log(Math.Max(p[i,j],1e-12));
    return loss/p.RowCount+lam*w.PointwiseMultiply(w).RowSums().Sum()/2;
}
(Matrix<double>, Vector<double>) Gradient(Matrix<double> x, Matrix<double> y, Matrix<double> w, Vector<double> b, double lam) {
    var dz=(Softmax(Scores(x,w,b))-y)/x.RowCount;
    return (x.TransposeThisAndMultiply(dz)+lam*w, dz.ColumnSums());
}
(Matrix<double>,Vector<double>,double[]) Entrainer(Matrix<double> x, Matrix<double> y, double lam, int iterations=1200) {
    var normal=new Normal(0,0.01,new Random(0)); var w=M.Dense(x.ColumnCount,y.ColumnCount,(i,j)=>normal.Sample());
    var b=V.Dense(y.ColumnCount); var hist=new double[iterations+1];
    for(int t=0;t<iterations;t++) { hist[t]=Perte(x,y,w,b,lam); var (gw,gb)=Gradient(x,y,w,b,lam); w-=0.5*gw; b-=0.5*gb; }
    hist[iterations]=Perte(x,y,w,b,lam); return (w,b,hist);
}
int[] Predire(Matrix<double> x, Matrix<double> w, Vector<double> b) => Softmax(Scores(x,w,b)).EnumerateRows().Select(r=>r.MaximumIndex()).ToArray();
var sx=M.DenseOfArray(new double[,]{{1,2},{-1,0.5},{0.4,-0.2}}); var sy=M.DenseOfArray(new double[,]{{1,0},{0,1},{1,0}});
var sw=M.DenseOfArray(new double[,]{{0.1,-0.2},{0.3,0.4}}); var sb=V.DenseOfArray(new[]{0.2,-0.1});
var (gw0,gb0)=Gradient(sx,sy,sw,sb,0.03); double eps=1e-6, err=0;
for(int i=0;i<2;i++) for(int j=0;j<2;j++) { var a=sw.Clone(); a[i,j]+=eps; var c=sw.Clone(); c[i,j]-=eps; err=Math.Max(err,Math.Abs(gw0[i,j]-(Perte(sx,sy,a,sb,0.03)-Perte(sx,sy,c,sb,0.03))/(2*eps))); }
for(int j=0;j<2;j++) { var a=sb.Clone(); a[j]+=eps; var c=sb.Clone(); c[j]-=eps; err=Math.Max(err,Math.Abs(gb0[j]-(Perte(sx,sy,sw,a,0.03)-Perte(sx,sy,sw,c,0.03))/(2*eps))); }
var stable=Softmax(M.DenseOfArray(new double[,]{{1000,1001},{0,1}}));
if(err>1e-7 || Math.Abs(stable[0,0]-stable[1,0])>1e-12 || Math.Abs(stable.Row(0).Sum()-1)>1e-12) throw new Exception("Contrôle gradient/softmax échoué");
Console.WriteLine($"Gradient W et b : erreur maximale {err:E3}; softmax stable et invariante vérifiée");

Gradient W et b : erreur maximale 5.340E-011; softmax stable et invariante vérifiée


Le gradient et la stabilité numérique sont contrôlés sur une petite fixture. Entraînons maintenant le modèle sur le banc complet.

In [5]:
var (W,b,historique)=Entrainer(XTrain,YTrain,0.001);
foreach(int it in new[]{0,25,50,100,200,400,800,1200}) Console.WriteLine($"{it,5} {historique[it]:F6}");
Console.WriteLine($"Perte initiale {historique[0]:F6}, finale {historique[^1]:F6}, réduction {historique[0]/historique[^1]:F2}x, variation sur 100 itérations {Math.Abs(historique[^1]-historique[^101]):E3}");

    0 1.386771


   25 1.364311


   50 1.348076


  100 1.317785


  200 1.264993


  400 1.184714


  800 1.090999


 1200 1.045848


Perte initiale 1.386771, finale 1.045848, réduction 1.33x, variation sur 100 itérations 8.447E-003


### Lecture de la convergence

La perte passe de 1,386771 à 1,045848 en 1 200 itérations, soit une réduction de 1,33 fois ; elle change encore de 0,008447 sur les 100 dernières. Le départ est proche de `log 4` car les poids sont petits. Le pas et le budget conditionnent la baisse observée : la convexité seule ne garantit pas une décroissance à chaque itération. Une perte résiduelle ne démontre pas l'absence de surapprentissage.

In [6]:
double Accuracy(int[] truth,int[] pred) => (double)truth.Where((x,i)=>x==pred[i]).Count()/truth.Length;
var predTrain=Predire(XTrain,W,b); var predTest=Predire(XTest,W,b);
var accTrain=Accuracy(yTrain,predTrain); var accTest=Accuracy(yTest,predTest);
Console.WriteLine($"Baseline {baseline:P2} | train {accTrain:P2} | test {accTest:P2} | gain {(accTest-baseline)*100:F2} points | écart train-test {(accTrain-accTest)*100:F2} points");
for(int c=0;c<4;c++) { var row=Enumerable.Range(0,4).Select(j=>yTest.Where((y,i)=>y==c && predTest[i]==j).Count()).ToArray();
    Console.WriteLine($"{categories[c],-8} {string.Join(" ",row)} | rappel {(double)row[c]/row.Sum():P2}"); }

Baseline 20.83 % | train 84.03 % | test 83.33 % | gain 62.50 points | écart train-test 0.69 points


sport    18 4 0 1 | rappel 78.26 %


tech     1 22 0 0 | rappel 95.65 %


science  2 3 22 3 | rappel 73.33 %


finance  1 1 0 18 | rappel 90.00 %


### Lecture du résultat

La précision de test est de 83,33 %, contre 20,83 % pour la baseline, soit un gain de 62,50 points ; l'entraînement atteint 84,03 %, un écart de 0,69 point. La matrice donne les rappels par classe : science est la plus faible (22/30, soit 73,33 %), avec trois confusions vers tech et trois vers finance. La précision globale masque ces différences. Un seul découpage ne suffit pas à établir la généralisation.

### Exercice 1 — Mini-lots

Implémentez la descente par mini-lots : mélangez les indices à chaque époque, calculez `P - Y` sur le seul lot et mettez à jour poids et biais. Mesurez la perte atteinte par lots de 16 contre la descente pleine passe pour un même nombre de documents vus. Les deux fixtures inversent les étiquettes et vérifient que l'historique final correspond au modèle rendu.

In [7]:
// Exercice 1 : mini-lots et comparaison à budget de documents vus égal.
(Matrix<double>,Vector<double>,double[])? EntrainerMiniLots(Matrix<double> x, Matrix<double> y,
    int taille=16, double lam=0.001, double lr=0.5, int epochs=50, int graine=0) {
    // TODO : mélanger chaque époque et calculer le gradient sur chaque lot.
    return null;
}
var xEx=M.DenseOfArray(new double[,]{{1,0},{0,1},{1,0},{0,1}});
var yEx=M.DenseOfArray(new double[,]{{1,0},{0,1},{1,0},{0,1}});
var exercice1=EntrainerMiniLots(xEx,yEx,taille:2);
if(exercice1==null) Console.WriteLine("Exercice 1 à compléter");
else {
    bool ValiderMini(Matrix<double> cible) {
        var r=EntrainerMiniLots(xEx,cible,taille:2);
        if(!r.HasValue) return false;
        var (w,bias,hist)=r.Value;
        return w.RowCount==2 && w.ColumnCount==2 && bias.Count==2 && hist.Length>=2 &&
            w.Enumerate().All(double.IsFinite) && bias.All(double.IsFinite) && hist.All(double.IsFinite) &&
            hist[^1]<hist[0] && Math.Abs(hist[^1]-Perte(xEx,cible,w,bias,0.001))<1e-8 && Predire(xEx,w,bias).SequenceEqual(cible.EnumerateRows().Select(v=>v.MaximumIndex()));
    }
    var inverse=M.DenseOfArray(new double[,]{{0,1},{1,0},{0,1},{1,0}});
    Console.WriteLine(ValiderMini(yEx) && ValiderMini(inverse) ? "Fixtures mini-lots validées" : "Réponse incorrecte");
}

Exercice 1 à compléter


## 4. Cross-check SOTA — la machine from scratch contre `scikit-learn`

`LogisticRegression` reçoit la même matrice et les mêmes étiquettes. La précision, l'accord par document, le cosinus et les normes des coefficients décrivent la ressemblance des résultats. Les régularisations et budgets diffèrent : ces mesures ne prouvent ni l'identité de l'objectif, ni celle de l'optimum. Le contrôle numérique indépendant du gradient, effectué ci-dessus, vérifie une propriété distincte de l'implémentation.

In [8]:
if (!PythonEngine.IsInitialized)
{
    var pythonDll = Environment.GetEnvironmentVariable("PYTHONNET_PYDLL");
    if (string.IsNullOrWhiteSpace(pythonDll))
        throw new InvalidOperationException("Définissez PYTHONNET_PYDLL avant le démarrage de Jupyter.");
    Runtime.PythonDLL = pythonDll;
    PythonEngine.Initialize();
    PythonEngine.BeginAllowThreads();
}
string[] skPred; double[][] coefficients; string[] skClasses;
using(Py.GIL()) {
    dynamic np=Py.Import("numpy"); dynamic mod=Py.Import("sklearn.linear_model"); dynamic sk=Py.Import("sklearn"); dynamic sys=Py.Import("sys");
    PyList ToPy(Matrix<double> x) { var outer=new PyList(); for(int i=0;i<x.RowCount;i++) { using var row=new PyList(); for(int j=0;j<x.ColumnCount;j++) { using var val=new PyFloat(x[i,j]); row.Append(val); } outer.Append(row); } return outer; }
    using var trainPy=ToPy(XTrain); using var testPy=ToPy(XTest);
    dynamic clf=mod.LogisticRegression(C:1000.0,max_iter:5000); clf.fit(np.array(trainPy),yTrain.Select(i=>categories[i]).ToArray().ToPython());
    using PyObject pred=clf.predict(np.array(testPy)); skPred=pred.InvokeMethod("tolist").As<string[]>();
    using PyObject cls=clf.classes_; skClasses=cls.InvokeMethod("tolist").As<string[]>();
    using PyObject coef=clf.coef_; coefficients=coef.InvokeMethod("tolist").As<double[][]>();
    Console.WriteLine($"Python {sys.version} | scikit-learn {sk.__version__}");
}
var skW=M.Dense(W.RowCount,W.ColumnCount,(i,j)=>coefficients[Array.IndexOf(skClasses,categories[j])][i]);
double cos=W.PointwiseMultiply(skW).RowSums().Sum()/(W.FrobeniusNorm()*skW.FrobeniusNorm());
Console.WriteLine($"Test C# {accTest:P2} | sklearn {(double)skPred.Where((x,i)=>x==categories[yTest[i]]).Count()/yTest.Length:P2} | baseline {baseline:P2}");
int accord=skPred.Where((x,i)=>x==categories[predTest[i]]).Count();
Console.WriteLine($"Accord {accord}/{yTest.Length} = {(double)accord/yTest.Length:P2}, cosinus {cos:F4}, normes {W.FrobeniusNorm():F4} / {skW.FrobeniusNorm():F4}");

Python 3.13.16 (main, Oct 14 2026, 01:04:01) [Clang 23.1.3 ] | scikit-learn 1.8.0


Test C# 83.33 % | sklearn 84.38 % | baseline 20.83 %


Accord 91/96 = 94.79 %, cosinus 0.7601, normes 13.4975 / 150.6573


<null>

### Lecture du cross-check

Sur les mêmes matrices, C# atteint 83,33 % et scikit-learn 84,38 % de précision ; leurs prédictions s'accordent sur 91 des 96 documents (94,79 %). Des précisions proches peuvent ainsi masquer des désaccords individuels. Le cosinus vaut 0,7601 et les normes 13,50 et 150,66 : les directions se ressemblent partiellement et les échelles diffèrent. Les budgets et régularisations diffèrent ; cette comparaison ne prouve pas un même optimum. Le contrôle par différences finies vérifie séparément le gradient.

### Exercice 2 — Banc déséquilibré

Construisez un banc déséquilibré à partir du générateur (par exemple 120 documents pour une classe et 20 pour chacune des autres), réentraînez le modèle puis calculez le **rappel** par classe : documents correctement prédits divisés par le support de chaque classe. Comparez ces rappels à la précision globale et expliquez ce qu'elle masque. Les fixtures contrôlent deux distributions de prédictions.

In [9]:
// Exercice 2 : rappel par classe sur données déséquilibrées.
Dictionary<string,double> RappelParClasse(string[] vrai,string[] pred,string[] classes) {
    // TODO : justes / support par classe.
    return null;
}
var exercice2=RappelParClasse(new[]{"a","a","b"},new[]{"a","b","b"},new[]{"a","b"});
if(exercice2==null) Console.WriteLine("Exercice 2 à compléter");
else {
    var autre=RappelParClasse(new[]{"a","b","b"},new[]{"b","a","a"},new[]{"a","b"});
    bool ValiderRappel(Dictionary<string,double> r,double a,double b) =>
        r != null && r.Count==2 && r.TryGetValue("a",out var x) && r.TryGetValue("b",out var y) &&
        double.IsFinite(x) && double.IsFinite(y) && Math.Abs(x-a)<1e-9 && Math.Abs(y-b)<1e-9;
    Console.WriteLine(ValiderRappel(exercice2,0.5,1) && ValiderRappel(autre,0,0)
        ? "Fixtures rappel validées" : "Réponse incorrecte");
}

Exercice 2 à compléter


## 5. Ce que la régularisation fait — et ce qu'elle ne fait pas

`lambda` fixe le prix des grands poids. La théorie annonce trois régimes :

- **`lambda = 0`** : aucun prix. Les poids peuvent devenir grands pour séparer des documents presque identiques, et le modèle colle alors au bruit de son jeu d'apprentissage ;
- **`lambda` trop grand** : le prix écrase tout. Les poids restent près de zéro, le modèle prédit presque uniformément et rejoint la baseline ;
- **entre les deux** : le compromis, qu'on repère en regardant l'**écart** entre précision d'apprentissage et précision de test, pas la seule précision de test.

Ces trois régimes sont un raisonnement, pas un résultat. Ce carnet ne les récite pas : il les **met à l'épreuve** sur son propre banc. La cellule suivante balaie `lambda` à réglages d'optimisation constants (`lr`, itérations, graine) — sans cette discipline, on ne saurait pas ce qui a produit la différence — et affiche, pour chaque valeur, la norme des poids, les deux précisions et leur écart.

Le résultat peut contredire l'attendu. C'est arrivé, et la lecture qui suit le dit tel quel.

In [10]:
var resultats = new List<(double Lam,double Train,double Test)>();
foreach(double lam in new[]{0.0,0.0001,0.001,0.01,0.1,1.0}) {
    var (wl,bl,hl)=Entrainer(XTrain,YTrain,lam);
    double a=Accuracy(yTrain,Predire(XTrain,wl,bl)), t=Accuracy(yTest,Predire(XTest,wl,bl));
    resultats.Add((lam,a,t));
    Console.WriteLine($"lambda={lam,7:F4} perte={hl[^1]:F6} norme={wl.FrobeniusNorm():F3} train={a:P2} test={t:P2} écart={a-t:P2}");
}
var maxTest=resultats.OrderByDescending(r=>r.Test).First();
var minEcart=resultats.OrderBy(r=>r.Train-r.Test).First();
Console.WriteLine($"Maximum descriptif sur test : lambda={maxTest.Lam:F4} ; minimum d'écart : lambda={minEcart.Lam:F4}");
Console.WriteLine("Ne pas choisir lambda sur le jeu de test : utiliser une validation séparée.");

lambda= 0.0000 perte=0.857541 norme=17.506 train=87.50 % test=83.33 % écart=4.17 %


lambda= 0.0001 perte=0.882834 norme=17.033 train=87.50 % test=83.33 % écart=4.17 %


lambda= 0.0010 perte=1.045848 norme=13.498 train=84.03 % test=83.33 % écart=0.69 %


lambda= 0.0100 perte=1.316833 norme=3.433 train=68.75 % test=58.33 % écart=10.42 %


lambda= 0.1000 perte=1.374039 norme=0.370 train=27.78 % test=20.83 % écart=6.94 %


lambda= 1.0000 perte=1.380237 norme=0.037 train=27.78 % test=20.83 % écart=6.94 %


Maximum descriptif sur test : lambda=0.0000 ; minimum d'écart : lambda=0.0010


Ne pas choisir lambda sur le jeu de test : utiliser une validation séparée.


### Lecture du balayage

La précision de test vaut 83,33 % pour les trois premières valeurs de `lambda`, puis 58,33 % à 0,01 et 20,83 % pour 0,1 et 1 ; la norme passe de 17,506 à 0,037. La pénalité forte réduit les poids et, sur ce banc, dégrade la précision. L'écart train-test passe de 4,17 points à zéro pénalité à 0,69 point pour 0,001, puis à 10,42 points pour 0,01 : l'écart seul ne suffit pas à juger la qualité. Ces mesures décrivent ce seul découpage, pas une loi générale : L2 peut aider ou nuire selon les données et l'optimisation, même lorsque les documents sont plus nombreux que les caractéristiques. Utiliser une validation séparée pour choisir un hyperparamètre.

### Exercice 3 — Bigrammes et régularisation

Ajoutez à chaque liste d'unigrammes les paires consécutives, séparées par `_`. Construisez le vocabulaire sur ces jetons et leur matrice en fréquences relatives avec le chemin pré-tokenisé (`tokenizer=lambda tokens: tokens` en Python ; `ConstruireMatriceTokens` en C#), sans retokeniser les bigrammes. Réentraînez puis balayez `lambda` aux mêmes réglages ; comparez la forme obtenue au tableau précédent sans supposer d'avance l'effet de L2.

In [11]:
// Exercice 3 : unigrammes suivis de bigrammes consécutifs.
List<string[]> Bigrammes(string[] docs) {
    // TODO : tokeniser, puis ajouter chaque paire adjacente séparée par _.
    // ConstruireMatriceTokens conserve les bigrammes sans les retokeniser.
    return null;
}
var exercice3=Bigrammes(new[]{"le match match", "but", ""});
if(exercice3==null) Console.WriteLine("Exercice 3 à compléter");
else Console.WriteLine(exercice3.Count==3 &&
    exercice3[0].SequenceEqual(new[]{"le","match","match","le_match","match_match"}) &&
    exercice3[1].SequenceEqual(new[]{"but"}) && exercice3[2].Length==0
    ? "Fixture bigrammes validée" : "Réponse incorrecte");
if(exercice3 != null && exercice3.Count==3 &&
    exercice3[0].SequenceEqual(new[]{"le","match","match","le_match","match_match"}) &&
    exercice3[1].SequenceEqual(new[]{"but"}) && exercice3[2].Length==0) {
    var jetons=Bigrammes(documents.ToArray());
    int taille=jetons.SelectMany(x=>x).Distinct().Count();
    Console.WriteLine($"Vocabulaire unigrammes : {vocabulaire.Length} | unigrammes et bigrammes : {taille}");
    Console.WriteLine($"Rapport caractéristiques / documents train : {(double)taille/idxTrain.Length:F1}");
}

Exercice 3 à compléter


## 6. Synthèse

Le modèle utilise des fréquences relatives sur 138 mots, une softmax et une descente de gradient pleine passe. Le contrôle par différences finies porte sur poids et biais, et le corpus synthétique conserve le protocole de 240 documents dont 144 pour l'entraînement.

Sur ce découpage C#, la perte passe de 1,386771 à 1,045848 ; la précision test atteint 83,33 % contre 20,83 % pour la baseline. Le témoin scikit-learn atteint 84,38 % et s'accorde sur 91/96 documents : la comparaison est informative mais ne démontre pas un optimum commun sous des réglages différents.

Le balayage montre une réduction de norme de 17,506 à 0,037 et une précision test allant de 83,33 % à 20,83 %. Ce résultat ne décrit qu'un split et un budget finis. Choisir `lambda` demande une validation distincte. Comparer Naive Bayes et régression logistique sur ce banc à représentation identique reste une expérience à réaliser.

## 7. Exercices

Les exercices répartis portent sur les [mini-lots](#exercice-1), le [rappel par classe](#exercice-2) et les [bigrammes](#exercice-3). Ils prolongent l'optimisation, l'évaluation d'un banc déséquilibré et l'étude de la régularisation.

## Références

- D. Jurafsky et J. H. Martin, *Speech and Language Processing*, 3e édition (draft), chapitre « Logistic Regression » — la dérivation de la softmax, de l'entropie croisée et de son gradient y est traitée sous l'angle du TAL.
- C. D. Manning, P. Raghavan et H. Schütze, *Introduction to Information Retrieval*, Cambridge University Press, 2008, chapitre 13 « Text classification and Naive Bayes » — le cadre génératif/discriminatif dont ce carnet et le carnet 10 occupent les deux pôles.
- `scikit-learn`, documentation de `sklearn.linear_model.LogisticRegression` — conventions de régularisation (`C`, multi-classe), utilisées ici comme juge extérieur.

## Navigation

- [<< Précédent — 10. Classification de texte Naive Bayes from scratch](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) — le pôle génératif du duo, dont la section 1 de ce carnet reprend le corpus comme socle de vocabulaire.
- [Index de la série](README.md) — le même point d'entrée que les dix autres carnets de la série.
- Carnet suivant : **aucun à ce jour** — ce carnet ferme la série ; le prochain numéro libre sera fixé à la consolidation de la série.

## À retenir

- La softmax calcule les probabilités de classe et le gradient utilise `P - Y`.
- La précision de test est de 83,33 % contre 20,83 % pour la baseline ; la matrice de confusion détaille les classes.
- Le témoin scikit-learn atteint 84,38 % et s'accorde sur 94,79 % des exemples ; les budgets et régularisations diffèrent, donc ces chiffres ne prouvent pas un même optimum.
- Le balayage de L2 décrit ce découpage et ne remplace pas une validation.

## Vérifiez votre compréhension

1. Pourquoi la perte initiale est-elle proche de `log 4` ?
2. Que distingue l’accord des prédictions de la précision ?
3. Pourquoi le cosinus ne prouve-t-il pas l’identité des optima ?
4. Le nombre de caractéristiques suffit-il à prévoir l’effet de L2 ?

<details><summary>Réponses</summary>

1. Les petits poids produisent des probabilités presque uniformes.
2. Il indique si les deux modèles rendent le même verdict document par document.
3. Les budgets et régularisations diffèrent.
4. Non, l’effet dépend des données et de l’optimisation.

</details>

## Pour aller plus loin

**Carnet précédent** : [10-Classement-Texte-NaiveBayes-FromScratch.ipynb](10-Classement-Texte-NaiveBayes-FromScratch.ipynb) — le pôle génératif du même problème, dont ce carnet réutilise le corpus comme socle de vocabulaire. La comparaison génératif/discriminatif sur **ce** banc chevauchant, à représentation identique, est l'expérience naturelle que la synthèse désigne — et qu'elle laisse ouverte.

À ce jour, ce carnet ferme la série NLP.

- **Lectures** : Ng & Jordan, « On Discriminative vs. Generative Classifiers : Logistic Regression and Naive Bayes » (NIPS 2001) — le couple exact que les carnets 10 et 11 incarnent, et pourquoi le génératif devance le discriminatif sur les petits échantillons ; Jurafsky & Martin, *Speech and Language Processing* (3e éd., draft), ch. « Logistic Regression ».
- **Dans le dépôt** : [02_NGrammes_Modeles_De_Langue.ipynb](02_NGrammes_Modeles_De_Langue.ipynb) — l'autre tradition d'estimation de la série ; la série `RAG-et-Memoire-Semantique`, où la représentation en sac-de-mots devient une question de recherche d'information.